## DIY Spatial processing visualization script 
## 1. Data Setup

In [ ]:
from pathlib import Path
import os
import gzip
import subprocess
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from sklearn.neighbors import NearestNeighbors
import string
import math
import anndata as ad
import tifffile as tf
import scanpy as sc
import seaborn as sns
import squidpy as sq
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.spatial import cKDTree
from skimage import exposure, filters, measure, morphology, segmentation, util

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(6, 6))
sns.set_context("notebook", font_scale=1.2)


In [ ]:
INFECTED_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/"
    "output-XETG00224__0023902__APPPS1_infected_453f30__20240627__192344"
)

MOCK_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/"
    "output-XETG00224__0023902__APPPS1_mock_437f2__20240627__192344"
)

IF_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "MAIN LAB FOLDER (Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/20240627__192310__KAECH_AD_GBM_240627/"
    "IF after Xenium run/OME-TIFF images for Xenium Explorer/"
    "APPPS1_infectedIF.ome.tif"
)

for path in [INFECTED_DIR, MOCK_DIR, IF_PATH]:
    print(path.exists(), path)


## 2. Understand and inspect the Xenium files

In [ ]:
EXPECTED_FILES = [
    "experiment.xenium",
    "analysis_summary.html",
    "metrics_summary.csv",
    "gene_panel.json",
    "cell_feature_matrix.h5",
    "cells.parquet",
    "transcripts.parquet",
    "cell_boundaries.parquet",
    "nucleus_boundaries.parquet",
    "cells.zarr.zip",
    "transcripts.zarr.zip",
    "analysis.zarr.zip",
    "morphology.ome.tif",
]

def inventory_xenium(folder):
    rows = []

    for name in EXPECTED_FILES:
        path = folder / name

        rows.append({
            "file": name,
            "present": path.exists(),
            "size_GB": (
                round(path.stat().st_size / 1024**3, 3)
                if path.is_file()
                else np.nan
            ),
        })

    return pd.DataFrame(rows)

display(inventory_xenium(INFECTED_DIR))
display(inventory_xenium(MOCK_DIR))

## 3. Inspect and run QC

In [ ]:
infected_metrics_path = INFECTED_DIR / "metrics_summary.csv"
mock_metrics_path = MOCK_DIR / "metrics_summary.csv"
print(os.access(infected_metrics_path, os.R_OK))
print(os.access(mock_metrics_path, os.R_OK))


In [ ]:
infected_metrics = pd.read_csv(INFECTED_DIR / "metrics_summary.csv")
mock_metrics = pd.read_csv(MOCK_DIR / "metrics_summary.csv")

metrics = pd.concat(
    {
        "infected": infected_metrics,
        "mock": mock_metrics
    }
)

columns = [
    "num_cells_detected",
    "fraction_transcripts_decoded_q20",
    "fraction_transcripts_assigned",
    "median_genes_per_cell",
    "median_transcripts_per_cell"
]

display(metrics[columns])

## 4. Load Xenium data into Scanpy

In [ ]:
def load_xenium(folder, sample_name):
    adata = sc.read_10x_h5(folder / "cell_feature_matrix.h5")
    adata.var_names_make_unique()
    print(adata.var["feature_types"].value_counts())

    # Keep biological genes in the expression matrix
    is_gene = adata.var["feature_types"].eq("Gene Expression")
    adata = adata[:, is_gene].copy()

    # Load the Xenium cell table
    cells = pd.read_parquet(folder / "cells.parquet")
    cells = cells.set_index("cell_id")

    # Load the Xenium cell table
    missing = adata.obs_names.difference(cells.index)
    print("Missing cell metadata: ", len(missing))

    adata.obs = adata.obs.join(
        cells.reindex(adata.obs_names)
    )

    adata.obs["sample"] = sample_name
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy(float)
    return adata

adata_infected = load_xenium(INFECTED_DIR, "infected")
adata_mock = load_xenium(MOCK_DIR, "mock")

print(adata_infected)
print(adata_mock)

### Combine the samples

adata = ad.concat(
    [adata_infected, adata_mock],
    label = "sample",
    keys = ["infected", "mock"],
    index_unique = "-",
    join = "inner"
)

adata.layers["counts"] = adata.X.copy()

print(adata)


## 5. Calculate QC metrics


In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    percent_top=None,
    log1p=True,
    inplace=True)

control_columns = [
    "control_probe_counts",
    "genomic_control_counts",
    "control_codeword_counts",
    "unassigned_codeword_counts"
]

adata.obs["control_fraction"] = (
    adata.obs[control_columns].sum(axis=1) / adata.obs["total_counts"].clip(lower=1e-6)
)

adata.obs["nucleus_to_cell_area"] = (
    adata.obs["nucleus_area"] / adata.obs["cell_area"].clip(lower=1e-6)
)

qc_columns = ["total_counts", "n_genes_by_counts", "control_fraction", "cell_area", "nucleus_area", "nucleus_count"]

display(adata.obs.groupby("sample", observed=True)[qc_columns].describe().round(2))

### Visualize QC distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, column in zip(axes.flat, qc_columns):
    sns.violinplot(data = adata.obs, x="sample", y=column, cut=0, inner="quartile", ax=ax)
    if column in ["total_counts", "cell_area", "nucleus_area"]:
        ax.set_yscale("log")

plt.tight_layout()
plt.show()

## 6. Spatial QC visualization

In [ ]:
def plot_spatial_qc(adata_sample, color, ax, point_size=0.5):
    xy = adata_sample.obsm["spatial"]
    values = adata_sample.obs[color]

    scatter = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=point_size, cmap="viridis", rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    ax.set_title(f"{adata_sample.obs['sample'].iloc[0]}: {color}")
    plt.colorbar(scatter, ax=ax, fraction=0.03)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for row, sample in enumerate(["infected", "mock"]):
    sample_adata = adata[adata.obs["sample"].eq(sample)].copy()
    plot_spatial_qc(sample_adata, "total_counts", axes[row, 0])
    plot_spatial_qc(sample_adata, "n_genes_by_counts", axes[row, 1])

plt.tight_layout()
plt.show()

adata.obs[
    ["total_counts", "cell_area", "nucleus_area"]
].groupby(adata.obs["sample"]).describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_data = adata.obs.copy()

# Remove zero values before using logarithmic axes
plot_data = plot_data[(plot_data["total_counts"] > 0)
    & (plot_data["cell_area"] > 0)
    & (plot_data["nucleus_area"] > 0)
]

sns.scatterplot(data=plot_data, x="total_counts", y="cell_area", hue="sample", s=7, alpha=0.25,linewidth=0, rasterized=True, ax=axes[0],)

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlim(plot_data["total_counts"].quantile(0.01), plot_data["total_counts"].quantile(0.99),)
axes[0].set_ylim(plot_data["cell_area"].quantile(0.01), plot_data["cell_area"].quantile(0.99),)
axes[0].set_title("Cell area vs. total counts")


sns.scatterplot(data=plot_data, x="nucleus_area", y="cell_area", hue="sample", s=7, alpha=0.25, linewidth=0, rasterized=True, ax=axes[1] )

axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlim(plot_data["nucleus_area"].quantile(0.01), plot_data["nucleus_area"].quantile(0.99),)
axes[1].set_ylim(plot_data["cell_area"].quantile(0.01), plot_data["cell_area"].quantile(0.99),)
axes[1].set_title("Cell area vs. nucleus area")
axes[1].get_legend().remove()

plt.tight_layout()
plt.show()


## 4. Define QC filters

### Perform QC separately on each sample

In [ ]:
def qc_one_sample(adata_sample, sample_name, min_counts=20, min_genes=10, max_control_fraction=0.1, 
                  area_lower_quantile=0.005, area_upper_quantile=0.995):
    a = adata_sample.copy()
    # Calculate the expression of QC independently
    sc.pp.calculate_qc_metrics(a, percent_top=None, inplace=True)
    control_columns = ["control_probe_counts", "genomic_control_counts", "control_codeword_counts", "unassigned_codeword_counts"]
    a.obs["control_fraction"] = (a.obs[control_columns].sum(axis=1) / a.obs["total_counts"].clip(lower=1))

    area_low = a.obs["cell_area"].quantile(area_lower_quantile)
    area_high = a.obs["cell_area"].quantile(area_upper_quantile)
    a.obs["counts_pass"] = (a.obs["total_counts"] >= min_counts)
    a.obs["genes_pass"] = (a.obs["n_genes_by_counts"] >= min_genes)
    a.obs["control_pass"] = (a.obs["control_fraction"].fillna(np.inf) <= max_control_fraction)
    a.obs["area_pass"] = (a.obs["cell_area"].between(area_low, area_high))
    a.obs["qc_pass"] = (a.obs["counts_pass"] & a.obs["genes_pass"] & a.obs["control_pass"] & a.obs["area_pass"])
    summary = a.obs[["counts_pass", "genes_pass", "control_pass", "area_pass", "qc_pass"]]
    print(f"\n{sample_name}")
    print(f"Area limits: {area_low:.2f}–{area_high:.2f} µm²")
    print(f"Minimum counts: {min_counts}")
    print(f"Minimum genes: {min_genes}")

    summary = (a.obs[["counts_pass", "genes_pass", "control_pass", "area_pass", "qc_pass"]].mean().mul(100).round(2))
    display(summary.to_frame("percent_passing"))
    print(
        f"Passed: {a.obs['qc_pass'].sum():,} / {a.n_obs:,} "
        f"({100 * a.obs['qc_pass'].mean():.2f}%)"
    )
    return a
    
infected_qc = qc_one_sample(
    adata_infected,
    sample_name="infected",
    min_counts=20,
    min_genes=10,
)

mock_qc = qc_one_sample(
    adata_mock,
    sample_name="mock",
    min_counts=20,
    min_genes=10,
)


### Plot the distributions separately

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (sample_name, a) in enumerate([("infected", infected_qc), ("mock", mock_qc),]):
    ## Histograms with total_counts
    sns.histplot(data=a.obs, x="total_counts", bins=100, ax=axes[row, 0])
    axes[row, 0].axvline(20, color="red", linestyle="--", label="QC threshold")
    axes[row, 0].set_xlim(0, a.obs["total_counts"].quantile(0.99),)
    axes[row, 0].set_title(f"{sample_name}: total counts")
    axes[row, 0].legend()

    ## Histograms with n_genes_by_counts
    sns.histplot(data=a.obs, x="n_genes_by_counts", bins=100, ax=axes[row, 1])
    axes[row, 1].axvline(10, color="red", linestyle="--", label="QC threshold")
    axes[row, 1].set_xlim(0, a.obs["n_genes_by_counts"].quantile(0.99),)
    axes[row, 1].set_title(f"{sample_name}: detected genes")
    axes[row, 1].legend()

plt.tight_layout()
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(13, 12),)
for row, (sample_name, a) in enumerate([("infected", infected_qc), ("mock", mock_qc)]):
    xy = a.obsm["spatial"]
    passed = a.obs["qc_pass"].to_numpy()
    axes[row, 0].scatter(xy[passed, 0], xy[passed, 1], s=0.25, c="steelblue", alpha=0.4, rasterized=True)
    axes[row, 0].set_title(f"{sample_name}: QC pass ({passed.sum():,})")

    axes[row, 1].scatter(xy[~passed, 0], xy[~passed, 1], s=0.5, c="red", alpha=0.5, rasterized=True)
    axes[row, 1].set_title(f"{sample_name}: QC fail ({(~passed).sum():,})")
    for ax in axes[row]:
        ax.invert_yaxis()
        ax.set_aspect("equal")
        ax.set_xlabel("x (µm)")
        ax.set_ylabel("y (µm)")

plt.tight_layout()
plt.show()

### Final QC

In [ ]:
comparison = pd.DataFrame(
    {
        "infected_standard": [
            infected_qc.obs["qc_pass"].sum(),
            infected_qc.obs["qc_pass"].mean() * 100,
        ],
        "mock_standard": [
            mock_qc.obs["qc_pass"].sum(),
            mock_qc.obs["qc_pass"].mean() * 100,
        ]
    },
    index=["cells_retained", "percent_retained"],
)

display(comparison.round(2))

infected_filtered = infected_qc[infected_qc.obs["qc_pass"]].copy()
mock_filtered = mock_qc[mock_qc.obs["qc_pass"]].copy()

# Make sure sample labels are present
infected_filtered.obs["sample"] = "infected"
mock_filtered.obs["sample"] = "mock"

adata_infected = infected_filtered.copy()
adata_mock = mock_filtered.copy()

adata_infected.layers["counts"] = (adata_infected.X.copy())
adata_mock.layers["counts"] = (adata_mock.X.copy())


## 8. Basic Scanpy Analysis

In [ ]:
def run_scanpy_processing(adata_sample, sample_name, leiden_resolution=0.5,):
    a = adata_sample.copy()
    # Normalize independently within this sample.
    sc.pp.normalize_total(a,target_sum=1e4,)
    sc.pp.log1p(a)
    # Store log-normalized expression before scaling.
    a.raw = a.copy()
    # Scale independently.
    sc.pp.scale(a, max_value=10)
    # PCA independently.
    sc.tl.pca(a, n_comps=50, svd_solver="arpack",)
    # Expression-neighbor graph independently.
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=30)
    # UMAP independently.
    sc.tl.umap(a, random_state=0,)

    # Clustering independently.
    sc.tl.leiden(a, resolution=leiden_resolution, key_added="leiden", random_state=0,)
    a.uns["analysis_sample"] = sample_name
    return a


adata_infected_processed = run_scanpy_processing(
    adata_infected,
    sample_name="infected",
    leiden_resolution=0.5,
)

adata_mock_processed = run_scanpy_processing(
    adata_mock,
    sample_name="mock",
    leiden_resolution=0.5,
)

## 9. Inspect PCA separately

In [ ]:

sc.pl.pca_variance_ratio(adata_infected_processed, n_pcs=50, log=True)
sc.pl.pca_variance_ratio(adata_mock_processed, n_pcs=50, log=True)


## 10. Plot UMAP

In [ ]:
fix, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata_infected_processed, color="leiden",
           legend_loc="on data", title="Infected", ax=axes[0], show=False)
sc.pl.umap(adata_mock_processed, color="leiden", 
           legend_loc="on data", title="Mock", ax=axes[1], show=False)

plt.tight_layout()
plt.show()

QC colored UMAPs

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11),)

sc.pl.umap(
    adata_infected_processed,
    color="total_counts",
    title="Infected: total counts",
    ax=axes[0, 0],
    show=False,
)

sc.pl.umap(
    adata_infected_processed,
    color="cell_area",
    title="Infected: cell area",
    ax=axes[0, 1],
    show=False,
)

sc.pl.umap(
    adata_mock_processed,
    color="total_counts",
    title="Mock: total counts",
    ax=axes[1, 0],
    show=False,
)

sc.pl.umap(
    adata_mock_processed,
    color="cell_area",
    title="Mock: cell area",
    ax=axes[1, 1],
    show=False,
)

plt.tight_layout()
plt.show()

### 11. Find cluster marker genes separately 

In [ ]:

sc.tl.rank_genes_groups(adata_infected_processed, groupby="leiden", method="wilcoxon", use_raw=True)
sc.pl.rank_genes_groups(adata_infected_processed, n_genes=10, sharey=False)

In [ ]:

sc.tl.rank_genes_groups(adata_mock_processed, groupby="leiden", method="wilcoxon", use_raw=True,)
sc.pl.rank_genes_groups(adata_mock_processed, n_genes=10, sharey=False,)

In [ ]:
infected_markers = sc.get.rank_genes_groups_df(adata_infected_processed, group=None)

mock_markers = sc.get.rank_genes_groups_df(adata_mock_processed, group=None)

display(infected_markers.head(20))
display(mock_markers.head(20))

## 12. Sqiudpy independently

In [ ]:
def run_squidpy_analysis(adata_sample, sample_name):
    a = adata_sample.copy()
    sq.gr.spatial_neighbors(a, coord_type="generic", n_neighs=8)
    sq.gr.nhood_enrichment(a, cluster_key="leiden", seed=0)
    sq.pl.nhood_enrichment(a, cluster_key="leiden", method="ward", title=f"{sample_name}: neighborhood enrichment")
    sq.gr.spatial_autocorr(a, mode="moran", genes=a.var_names.tolist(), n_perms=100)

    print(f"{sample_name}: top spatial genes")
    display(a.uns["moranI"].head(20))
    return a


In [ ]:

adata_infected_spatial = run_squidpy_analysis(adata_infected_processed, sample_name="infected")
adata_mock_spatial = run_squidpy_analysis(adata_mock_processed, sample_name="mock")


### save excel sheets and DEG heatmap to identify the clusters

In [ ]:
## Calculate the marker genes

sc.tl.rank_genes_groups(
    adata_infected_processed,
    groupby="leiden",
    method="wilcoxon",
    use_raw=True,
    key_added="cluster_markers",
)

sc.tl.rank_genes_groups(
    adata_mock_processed,
    groupby="leiden",
    method="wilcoxon",
    use_raw=True,
    key_added="cluster_markers",
)

## Create marker tables

infected_markers = sc.get.rank_genes_groups_df(adata_infected_processed,
    group=None,
    key="cluster_markers",
)

mock_markers = sc.get.rank_genes_groups_df(
    adata_mock_processed,
    group=None,
    key="cluster_markers",
)

# Sort by cluster and adjusted p-value.
infected_markers = infected_markers.sort_values(
    ["group", "pvals_adj", "scores"],
    ascending=[True, True, False],
)

mock_markers = mock_markers.sort_values(
    ["group", "pvals_adj", "scores"],
    ascending=[True, True, False],
)

display(infected_markers.head(20))
display(mock_markers.head(20))

## Add significance columns
def annotate_marker_table(markers):
    markers = markers.copy()
    markers["significant"] = ((markers["pvals_adj"] < 0.05)& (markers["logfoldchanges"] > 0.25))
    markers["minus_log10_padj"] = (-np.log10(markers["pvals_adj"].clip(lower=1e-300)))
    return markers


infected_markers = annotate_marker_table(infected_markers)
mock_markers = annotate_marker_table(mock_markers)
infected_significant = infected_markers[infected_markers["significant"]].copy()

mock_significant = mock_markers[mock_markers["significant"]].copy()

### Top 10 markers per cluster
infected_top10 = (
    infected_significant
    .sort_values(
        ["group", "scores"],
        ascending=[True, False],
    )
    .groupby("group", observed=True)
    .head(10)
)

mock_top10 = (
    mock_significant
    .sort_values(
        ["group", "scores"],
        ascending=[True, False],
    )
    .groupby("group", observed=True)
    .head(10)
)

from pathlib import Path

OUTPUT_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/"
    "Valisha/AD Serial Infection Project  (Irene & Brian W)/"
    "Spatial Transcriptomics 20260518/RESULTS/beginner_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MARKER_EXCEL = OUTPUT_DIR / "infected_mock_cluster_markers.xlsx"

with pd.ExcelWriter(MARKER_EXCEL,engine="openpyxl") as writer:
    infected_markers.to_excel(writer, sheet_name="infected_all", index=False)
    infected_significant.to_excel(writer, sheet_name="infected_significant", index=False,)
    infected_top10.to_excel(writer,sheet_name="infected_top10",index=False)
    mock_markers.to_excel(writer, sheet_name="mock_all", index=False,)
    mock_significant.to_excel(writer, sheet_name="mock_significant", index=False,)
    mock_top10.to_excel( writer, sheet_name="mock_top10", index=False,)

print("Saved:", MARKER_EXCEL)

sc.pl.rank_genes_groups_heatmap(
    adata_infected_processed,
    groupby="leiden",
    key="cluster_markers",
    n_genes=5,
    use_raw=True,
    standard_scale="var",
    swap_axes=True,
    dendrogram=False,  # do not cluster columns
    show_gene_labels=True,
    cmap="viridis",
    figsize=(14, 10),
    show=False,
)

plt.gcf().suptitle(
    "Infected: top marker genes",
    y=1.02,
)

plt.gcf().savefig(
    OUTPUT_DIR / "infected_cluster_marker_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

sc.pl.rank_genes_groups_heatmap(
    adata_mock_processed,
    groupby="leiden",
    key="cluster_markers",
    n_genes=5,
    use_raw=True,
    standard_scale="var",
    swap_axes=True,
    dendrogram=False,  # do not cluster columns
    show_gene_labels=True,
    cmap="viridis",
    figsize=(14, 10),
    show=False,
)

plt.gcf().suptitle(
    "Mock: top marker genes",
    y=1.02,
)

plt.gcf().savefig(
    OUTPUT_DIR / "mock_cluster_marker_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

sc.pl.rank_genes_groups_dotplot(
    adata_infected_processed,
    groupby="leiden",
    key="cluster_markers",
    n_genes=5,
    use_raw=True,
    standard_scale="var",
    cmap="Reds",
    show=False,
)

plt.gcf().savefig(
    OUTPUT_DIR / "infected_cluster_marker_dotplot.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

sc.pl.rank_genes_groups_dotplot(
    adata_mock_processed,
    groupby="leiden",
    key="cluster_markers",
    n_genes=5,
    use_raw=True,
    standard_scale="var",
    cmap="Blues",
    show=False,
)

plt.gcf().savefig(
    OUTPUT_DIR / "mock_cluster_marker_dotplot.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()


## 13. Plot clusters spatially

In [ ]:
def plot_spatial_clusters(adata_sample, sample_name, ax, point_size=0.5):
    xy = adata_sample.obsm["spatial"]
    clusters = (adata_sample.obs["leiden"].astype("category"))
    cluster_codes = clusters.cat.codes
    scatter = ax.scatter(xy[:, 0], xy[:, 1], c=cluster_codes, s=point_size, cmap="tab20", alpha=0.8, linewidth=0, rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    ax.set_title(f"{sample_name}: spatial clusters")

    return scatter

fig, axes = plt.subplots(1, 2, figsize=(15, 7),)

plot_spatial_clusters(adata_infected_spatial, "Infected", axes[0],)
plot_spatial_clusters(adata_mock_spatial, "Mock", axes[1],)

plt.tight_layout()
plt.show()

## 14. Load native Xenium morphology separately for Infected and Mock

In [ ]:
def read_ome_level(path, level=-1):
    with tf.TiffFile(path) as tif:
        series = tif.series[0]
        levels = list( getattr(series, "levels", None) or [series])
        if level < 0:
            level = len(levels) + level
        level = max(0, min(level, len(levels) - 1),)
        image = levels[level].asarray()
        information = {
            "axes": levels[level].axes,
            "level": level,
            "level_shape": levels[level].shape,
            "full_shape": levels[0].shape,
            "number_of_levels": len(levels),
        }
    return image, information

In [ ]:
infected_morphology, infected_morphology_info = (
    read_ome_level(
        INFECTED_DIR / "morphology.ome.tif",
        level=-1,
    )
)

mock_morphology, mock_morphology_info = (
    read_ome_level(
        MOCK_DIR / "morphology.ome.tif",
        level=-1,
    )
)

# Maximum projection over the Xenium DAPI Z-stack.
infected_dapi = infected_morphology.max(axis=0)
mock_dapi = mock_morphology.max(axis=0)

print("Infected:", infected_morphology_info)
print("Mock:", mock_morphology_info)

## 15. Overlay cells on native xenium images separately

In [ ]:
XENIUM_PIXEL_SIZE_UM = 0.2125

def cells_at_image_level(adata_sample, image_information):
    full_y, full_x = image_information["full_shape"][-2:]
    level_y, level_x = image_information["level_shape"][-2:]

    downsample_x = full_x / level_x
    downsample_y = full_y / level_y

    cell_xy_um = adata_sample.obsm["spatial"]

    cell_level_xy = np.column_stack(
        [
            (
                cell_xy_um[:, 0]
                / XENIUM_PIXEL_SIZE_UM
                / downsample_x
            ),
            (
                cell_xy_um[:, 1]
                / XENIUM_PIXEL_SIZE_UM
                / downsample_y
            ),
        ]
    )

    return cell_level_xy

In [ ]:

infected_cell_image_xy = cells_at_image_level(adata_infected_spatial, infected_morphology_info)
mock_cell_image_xy = cells_at_image_level(adata_mock_spatial, mock_morphology_info)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8),)
axes[0].imshow(infected_dapi, cmap="gray")

axes[0].scatter(
    infected_cell_image_xy[:, 0],
    infected_cell_image_xy[:, 1],
    s=0.15,
    c="lime",
    alpha=0.3,
    rasterized=True,
)

axes[0].set_title("Infected cells over infected Xenium morphology")
axes[0].set_xlim(0, infected_dapi.shape[1],)
axes[0].set_ylim(infected_dapi.shape[0], 0)

axes[1].imshow(mock_dapi, cmap="gray")

axes[1].scatter(
    mock_cell_image_xy[:, 0],
    mock_cell_image_xy[:, 1],
    s=0.15,
    c="cyan",
    alpha=0.3,
    rasterized=True,
)

axes[1].set_title("Mock cells over mock Xenium morphology")
axes[1].set_xlim(0, mock_dapi.shape[1],)

axes[1].set_ylim(mock_dapi.shape[0], 0)

for ax in axes:
    ax.set_aspect("equal")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 16. Inspect the combined post-xenium IF image

In [ ]:
if_thumbnail, if_information = read_ome_level(IF_PATH, level=-1,)

fig, axes = plt.subplots(2, 2, figsize=(12, 10),)

for channel, ax in enumerate(axes.flat):
    image = if_thumbnail[channel]
    low, high = np.percentile( image, [1, 99.8],)
    displayed_image = exposure.rescale_intensity(image, in_range=(low, high),)
    ax.imshow( displayed_image, cmap="gray",)
    ax.set_title( f"IF channel index {channel}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 17. Affine-coordinate helper

In [ ]:
def apply_affine_xy(xy, affine_matrix):
    xy = np.asarray(xy, dtype=float,)
    homogeneous_xy = np.column_stack([xy,np.ones(len(xy)),])
    transformed = (homogeneous_xy @ affine_matrix.T)
    return (transformed[:, :2] / transformed[:, 2:3])

INFECTED_ALIGNMENT_MATRIX = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/"
    "Kaech Lab Folder/MAIN LAB FOLDER "
    "(Ordering, lab meetings, protocols, IACUC)/"
    "Spatial transcriptomics datasets/Post Xenium IF/"
    "APPPS1_infectedIF_alignment_files/matrix.csv"
)

A_INFECTED_IF_TO_XENIUM = np.loadtxt(INFECTED_ALIGNMENT_MATRIX, delimiter=",")

print(A_INFECTED_IF_TO_XENIUM)

## 19. Separate IF overlay function

In [ ]:

def overlay_sample_on_if(adata_sample, affine_if_to_xenium, if_thumbnail, if_information, channel, sample_name, color, ax):
    # Xenium micrometers -> full-resolution Xenium pixels.
    cell_xenium_pixels = (adata_sample.obsm["spatial"] / XENIUM_PIXEL_SIZE_UM)
    # Xenium pixels -> full-resolution IF pixels.
    cell_if_full_pixels = apply_affine_xy(cell_xenium_pixels, np.linalg.inv(affine_if_to_xenium))
    full_y, full_x = if_information["full_shape"][-2:]
    level_y, level_x = if_information["level_shape"][-2:]
    downsample_xy = np.array([full_x / level_x, full_y / level_y])
    cell_if_level_pixels = (cell_if_full_pixels / downsample_xy)
    image = if_thumbnail[channel]
    low, high = np.percentile(image,[1, 99.8])
    displayed_image = exposure.rescale_intensity(image, in_range=(low, high),)
    ax.imshow(displayed_image, cmap="gray")
    ax.scatter(
        cell_if_level_pixels[:, 0],
        cell_if_level_pixels[:, 1],
        s=0.2,
        c=color,
        alpha=0.35,
        linewidth=0,
        rasterized=True)
    ax.set_xlim(0, level_x)
    ax.set_ylim(level_y, 0)
    ax.set_aspect("equal")
    ax.set_title(f"{sample_name}: cells over post-Xenium IF")
    ax.axis("off")
    return cell_if_level_pixels


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 9)
)
# Zero-based channel index: 0, 1, 2, or 3
ALIGNMENT_CHANNEL = 2
# Change after confirming the staining records.
PLAQUE_CHANNEL = 2
infected_cell_if_xy = overlay_sample_on_if(
    adata_sample=adata_infected_spatial,
    affine_if_to_xenium=A_INFECTED_IF_TO_XENIUM,
    if_thumbnail=if_thumbnail,
    if_information=if_information,
    channel=PLAQUE_CHANNEL,
    sample_name="Infected",
    color="lime",
    ax=ax,
)
plt.show()

## 20. Identify Plaques

In [ ]:
## Select the plaque channel
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for channel, ax in enumerate(axes.flat):
    image = if_thumbnail[channel].astype(float)
    low, high = np.percentile(image, [1, 99.8])

    shown = exposure.rescale_intensity(image, in_range=(low, high),)

    ax.imshow(shown, cmap="gray")
    ax.set_title(f"Channel index {channel}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### 20a. Load a moderate-resolution IF level

In [ ]:

def cells_to_if_level(adata_sample, affine_if_to_xenium, image_transformation):
    cell_xenium_pixels = (adata_sample.obsm["spatial"] / XENIUM_PIXEL_SIZE_UM)
    # Xenium pixels -> full-resolution source IF pixels
    cell_if_full_pixels = apply_affine_xy(cell_xenium_pixels, np.linalg.inv(affine_if_to_xenium))
    full_y, full_x = image_transformation["full_shape"][-2:]
    level_y, level_x = image_transformation["level_shape"][-2:]
    downsample_xy = np.array([full_x / level_x, full_y / level_y])
    cell_if_level_pixels = (cell_if_full_pixels / downsample_xy)
    return cell_if_level_pixels, downsample_xy
PLAQUE_LEVEL = 2
if_plaque_image, if_plaque_information = read_ome_level(IF_PATH, level=PLAQUE_LEVEL)
plaque_image = if_plaque_image[PLAQUE_CHANNEL].astype(np.float32)
print("Image shape:", plaque_image.shape)
print(if_plaque_information)


### 20b. Define the infected tissue region

In [ ]:
from scipy.spatial import ConvexHull
from skimage.draw import polygon2mask

infected_cells_if_xy, if_downsample_xy = cells_to_if_level(adata_infected_spatial, A_INFECTED_IF_TO_XENIUM, if_plaque_information)
valid = (
    np.isfinite(infected_cells_if_xy).all(axis=1)
    & (infected_cells_if_xy[:, 0] >= 0)
    & (infected_cells_if_xy[:, 1] >= 0)
    & (infected_cells_if_xy[:, 0] < plaque_image.shape[1])
    & (infected_cells_if_xy[:, 1] < plaque_image.shape[0])
)

infected_cells_if_valid = infected_cells_if_xy[valid]
hull = ConvexHull(infected_cells_if_valid)
hull_xy = infected_cells_if_valid[hull.vertices]
infected_tissue_mask = polygon2mask(plaque_image.shape, hull_xy[:, [1, 0]])


In [ ]:
low, high = np.percentile(plaque_image[infected_tissue_mask], [1, 99.8],)

shown = exposure.rescale_intensity(plaque_image, in_range=(low, high),)

plt.figure(figsize=(11, 9))
plt.imshow(shown, cmap="gray")

plt.contour(
    infected_tissue_mask,
    levels=[0.5],
    colors="cyan",
    linewidths=1,
)

plt.scatter(
    infected_cells_if_valid[:, 0],
    infected_cells_if_valid[:, 1],
    s=0.1,
    c="lime",
    alpha=0.2,
)

plt.title("Infected tissue region used for plaque detection")
plt.axis("off")
plt.show()

### 20c. Test brighter plaque seeds

In [ ]:
tissue_values = plaque_image[infected_tissue_mask]

white_percentiles = {
    "p99.90": 99.90,
    "p99.95": 99.95,
    "p99.975": 99.975,
    "p99.99": 99.99,
}

white_masks = {}

for name, percentile in white_percentiles.items():
    threshold = np.percentile(tissue_values, percentile)
    mask = (plaque_image >= threshold) & infected_tissue_mask
    mask = morphology.closing(mask, morphology.disk(1))
    mask = morphology.remove_small_objects(mask, min_size=3)
    white_masks[name] = mask

    print(name, "threshold:", round(threshold, 1), "objects:", measure.label(mask).max())

### 20d. Compare the three settings

In [ ]:
low, high = np.percentile(tissue_values, [1, 99.9])
shown = exposure.rescale_intensity(plaque_image, in_range=(low, high))

for name, mask in white_masks.items():
    plt.figure(figsize=(11, 9))
    plt.imshow(shown, cmap="gray")
    plt.contour(mask, levels=[0.5], colors="cyan", linewidths=0.7)
    plt.title(f"{name}: {measure.label(mask).max()} objects")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:

XENIUM_PIXEL_SIZE_UM = 0.2125
IF_FULL_HEIGHT = 9285
IF_FULL_WIDTH = 10203

level_downsample_x = IF_FULL_WIDTH / plaque_image.shape[1]
level_downsample_y = IF_FULL_HEIGHT / plaque_image.shape[0]

affine_area_scale = abs(np.linalg.det(A_INFECTED_IF_TO_XENIUM[:2, :2]))
full_if_pixel_area_um2 = affine_area_scale * XENIUM_PIXEL_SIZE_UM ** 2
pixel_area_um2 = full_if_pixel_area_um2 * level_downsample_x * level_downsample_y
pixel_size_um = np.sqrt(pixel_area_um2)

print("Plaque image shape:", plaque_image.shape)
print("Level downsample:", round(level_downsample_x, 3), round(level_downsample_y, 3))
print("Pixel size:", round(pixel_size_um, 3), "µm")
print("Pixel area:", round(pixel_area_um2, 3), "µm²")

seed_percentile = 99.85
support_percentile = 99.20
growth_radius_pixels = 5

seed_threshold = np.percentile(tissue_values, seed_percentile)
support_threshold = np.percentile(tissue_values, support_percentile)

seed_mask = (plaque_image >= seed_threshold) & infected_tissue_mask
support_mask = (plaque_image >= support_threshold) & infected_tissue_mask

near_seed_mask = morphology.dilation(seed_mask, morphology.disk(growth_radius_pixels))
candidate_mask = support_mask & near_seed_mask
candidate_mask |= seed_mask

candidate_labels = measure.label(candidate_mask)

print("Bright seeds:", measure.label(seed_mask).max())
print("Grown candidates:", candidate_labels.max())

### Measure whether objects are compact spots
properties = measure.regionprops_table(
    candidate_labels,
    intensity_image=plaque_image,
    properties=("label", "area", "perimeter", "major_axis_length", "minor_axis_length",
                "eccentricity", "solidity", "mean_intensity", "max_intensity", "centroid"),
)

plaque_objects = pd.DataFrame(properties)
plaque_objects["area_um2"] = plaque_objects["area"] * pixel_area_um2
plaque_objects["diameter_um"] = 2 * np.sqrt(plaque_objects["area_um2"] / np.pi)
plaque_objects["major_axis_um"] = plaque_objects["major_axis_length"] * pixel_size_um
plaque_objects["axis_ratio"] = plaque_objects["minor_axis_length"] / plaque_objects["major_axis_length"].clip(lower=1)
plaque_objects["circularity"] = 4 * np.pi * plaque_objects["area"] / plaque_objects["perimeter"].clip(lower=1) ** 2

### Remove crescent/moon shapes
seed_labels = candidate_labels[seed_mask & (candidate_labels > 0)]
seed_counts = np.bincount(seed_labels, minlength=candidate_labels.max() + 1)
plaque_objects["seed_pixels"] = plaque_objects["label"].map(lambda label: seed_counts[int(label)])

### Remove tiny specs
MIN_PLAQUE_AREA_UM2 = 10
MAX_PLAQUE_AREA_UM2 = 10000

plaque_objects["too_small"] = plaque_objects["area_um2"] < MIN_PLAQUE_AREA_UM2
plaque_objects["too_large"] = plaque_objects["area_um2"] > MAX_PLAQUE_AREA_UM2
plaque_objects["too_narrow"] = plaque_objects["diameter_um"] < 3
plaque_objects["moon_like"] = moon_like

properties = measure.regionprops_table(
    candidate_labels,
    intensity_image=plaque_image,
    properties=("label", "area", "perimeter", "major_axis_length", "minor_axis_length",
                "eccentricity", "solidity", "mean_intensity", "max_intensity", "centroid"),
)

plaque_objects = pd.DataFrame(properties)
plaque_objects["area_um2"] = plaque_objects["area"] * pixel_area_um2
plaque_objects["diameter_um"] = 2 * np.sqrt(plaque_objects["area_um2"] / np.pi)
plaque_objects["major_axis_um"] = plaque_objects["major_axis_length"] * pixel_size_um
plaque_objects["axis_ratio"] = plaque_objects["minor_axis_length"] / plaque_objects["major_axis_length"].clip(lower=1)
plaque_objects["circularity"] = 4 * np.pi * plaque_objects["area"] / plaque_objects["perimeter"].clip(lower=1) ** 2

seed_labels = candidate_labels[seed_mask & (candidate_labels > 0)]
seed_counts = np.bincount(seed_labels, minlength=candidate_labels.max() + 1)
plaque_objects["seed_pixels"] = plaque_objects["label"].map(lambda label: seed_counts[int(label)])

moon_like = (
    (plaque_objects["area"] >= MIN_PLAQUE_AREA_UM2) &
    (plaque_objects["major_axis_um"] > MAX_PLAQUE_AREA_UM2) &
    (plaque_objects["axis_ratio"] < 0.20) &
    (plaque_objects["solidity"] < 0.45) &
    (plaque_objects["circularity"] < 0.12)
)

plaque_objects["keep"] = (
    (plaque_objects["seed_pixels"] >= 1) &
    (plaque_objects["area_um2"] <= 500) &
    ~moon_like
)

print("Candidates:", len(plaque_objects))
print("No bright seed:", (plaque_objects["seed_pixels"] < 1).sum())
print("Specific moon-like artifacts:", moon_like.sum())
print("Plaques retained:", plaque_objects["keep"].sum())

### Plot the filtered result
kept_labels = plaque_objects.loc[plaque_objects["keep"], "label"].to_numpy()
rejected_labels = plaque_objects.loc[~plaque_objects["keep"], "label"].to_numpy()

final_plaque_mask = np.isin(candidate_labels, kept_labels)
rejected_mask = np.isin(candidate_labels, rejected_labels)

low, high = np.percentile(tissue_values, [1, 99.9])
shown = exposure.rescale_intensity(plaque_image, in_range=(low, high))

plt.figure(figsize=(11, 9))
plt.imshow(shown, cmap="gray")
plt.contour(final_plaque_mask, levels=[0.5], colors="cyan", linewidths=0.8)
plt.contour(rejected_mask, levels=[0.5], colors="magenta", linewidths=0.4)
plt.title(f"Plaques retained: {plaque_objects['keep'].sum()}")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:

print("Plaques retained:", plaque_objects["keep"].sum())
print("Specific crescent artifacts removed:", moon_like.sum())

kept_labels = plaque_objects.loc[plaque_objects["keep"], "label"].to_numpy()
rejected_labels = plaque_objects.loc[~plaque_objects["keep"], "label"].to_numpy()

final_plaque_mask = np.isin(candidate_labels, kept_labels)
rejected_mask = np.isin(candidate_labels, rejected_labels)

plt.figure(figsize=(11, 9))
plt.imshow(shown, cmap="gray")
plt.contour(final_plaque_mask, levels=[0.5], colors="cyan", linewidths=0.8)

if rejected_mask.any():
    plt.contour(rejected_mask, levels=[0.5], colors="magenta", linewidths=0.7)

plt.title(f"Plaques retained: {plaque_objects['keep'].sum()}")
plt.axis("off")
plt.tight_layout()
plt.show()


# 20. Mock IF overlay

In [ ]:
# ALIGNMENT_CHANNEL = 2

# if_full_display, if_full_info = read_ome_level(
#     IF_PATH,
#     level=-1,
# )

# image = if_full_display[ALIGNMENT_CHANNEL]

# low, high = np.percentile(
#     image,
#     [1, 99.8],
# )

# image_display = exposure.rescale_intensity(
#     image,
#     in_range=(low, high),
# )

# plt.figure(figsize=(12, 10))
# plt.imshow(image_display, cmap="gray")
# plt.title("Combined IF: identify the mock tissue")
# plt.axis("off")
# plt.show()



In [ ]:
if_thumbnail